# Разведочный анализ данных: прогноз исхода цирроза печени

Датасет: Playground Series S3E26 (Mayo Clinic PBC)

Цель этого этапа — изучить данные, выявить аномалии, сформулировать гипотезы и утвердить план генерации признаков перед этапом моделирования.

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import phik
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('data/raw/train.csv')
print('Размер выборки:', df.shape)
df.head()

## 2. Качество данных

### 2.1. Пропуски

In [ ]:
missing = df.isna().sum().sum()
print('Пропущенных значений:', missing)

Пропусков в данных нет. Это упрощает этап предобработки — не нужно применять импутацию.

### 2.2. Дубликаты

In [ ]:
full_dups = df.duplicated().sum()
partial_dups = df.drop(columns=['id']).duplicated().sum()
print('Полных дубликатов:', full_dups)
print('Частичных дубликатов (без id):', partial_dups)

Полных дубликатов нет, но после удаления технического поля `id` обнаружено 2 частичных дубликата — пациенты с одинаковыми клиническими показателями. Их нужно удалить.

### 2.3. Роль поля `id`

Поле `id` — это просто порядковый номер строки. Оно не несёт клинической информации, поэтому в модель его подавать нельзя: алгоритм выучит привязку к конкретным номерам и переобучится.

При этом `id` нужно сохранить отдельно — он понадобится на финальном этапе, чтобы сопоставить предсказанные вероятности с пациентами при формировании файла сабмишна.

### 2.4. Анализ выбросов

In [ ]:
numeric_cols = ['Bilirubin', 'Prothrombin', 'Copper', 'Alk_Phos', 
                'SGOT', 'Tryglicerides', 'Cholesterol']

def get_margins(col):
    q1, q3 = col.quantile(0.25), col.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

results = []
for col in numeric_cols:
    left, right = get_margins(df[col])
    filtered = df[df[col] > right]
    counts = filtered['Status'].value_counts()
    total = counts.sum()
    if total > 0:
        results.append({
            'Feature': col,
            'Total': total,
            'D_percent': round(counts.get('D', 0) / total * 100, 1)
        })

pd.DataFrame(results).sort_values('D_percent', ascending=False)

Экстремальные значения в биохимических показателях на первый взгляд выглядят подозрительно. Но проверка распределения статусов среди пациентов с такими отклонениями показала:

- среди пациентов с аномально высоким билирубином 80.5% имели летальный исход;
- для протромбина и меди — 80.3% и 77.3% соответственно;
- пациенты с одновременно высокими билирубином, протромбином, медью и низким альбумином (< 2.72) в 100% случаев имели статус `D`.

Это реальные маркеры тяжёлого состояния, а не ошибки ввода. Удалять или winsorize их нельзя — потеряется важная информация.

### 2.5. Ловушка с возрастом

In [ ]:
print('Минимальный Age:', df['Age'].min())
print('Максимальный Age:', df['Age'].max())
print('Средний возраст в годах:', round(df['Age'].mean() / 365.25, 1))

Признак `Age` записан в днях (значения ~10 000–28 000), а не в годах. Если не разделить на 365.25, признак становится практически бесполезным, а медицинские формулы (например, Mayo Score) ломаются.

## 3. Одномерный анализ

### 3.1. Числовые признаки

In [ ]:
skewness = df[numeric_cols].skew().sort_values(ascending=False)
plt.figure(figsize=(10, 5))
skewness.plot(kind='barh', color='steelblue')
plt.axvline(x=0, color='red', linestyle='--')
plt.title('Асимметрия числовых признаков')
plt.xlabel('Skewness')
plt.tight_layout()
plt.show()

Анализ асимметрии показал:

- равномерное распределение у `Age`, `Platelets`, `N_Days`;
- незначительное левостороннее смещение у `Albumin` (трансформация не нужна);
- сильное правостороннее смещение у `Prothrombin`, `SGOT`, `Copper`, `Tryglicerides`, `Alk_Phos`, `Bilirubin`, `Cholesterol` — для них нужно логарифмирование.

### 3.2. Категориальные признаки

In [ ]:
categorical_cols = ['Drug', 'Sex', 'Ascites', 'Edema', 'Spiders', 'Hepatomegaly']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, hue='Status', ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

- `Drug` и `Hepatomegaly` равнопредставлены;
- `Sex` — преобладание женщин, что соответствует эпидемиологии PBC;
- `Ascites`, `Edema`, `Spiders` имеют сильный перекос и встречаются преимущественно у пациентов с негативным исходом.

### 3.3. Целевая переменная

In [ ]:
status_counts = df['Status'].value_counts()
print(status_counts)
print()
for cls in ['C', 'D', 'CL']:
    print(f'Доля класса {cls}: {status_counts[cls] / len(df) * 100:.1f}%')

Класс `C` существенно преобладает. Accuracy будет вводить в заблуждение, поэтому основными метриками выбираем F1-macro и LogLoss.

## 4. Корреляционный анализ

In [ ]:
phik_matrix = df.phik_matrix(interval_cols=numeric_cols)
plt.figure(figsize=(12, 10))
sns.heatmap(phik_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, vmin=0, vmax=1)
plt.title('Матрица корреляций (коэффициент phik)')
plt.tight_layout()
plt.show()

Матрица корреляций через коэффициент phik (учитывает нелинейные зависимости) показала:

- самая сильная связь с `Status` у `Copper` (0.60), `Edema` (0.56) и `Bilirubin` (0.50);
- коэффициент связи между `Drug` и `Status` практически нулевой (0.01) — препарат не даёт эффекта по сравнению с плацебо, что согласуется с клиническими данными;
- развитие асцита связано с падением уровня альбумина и ростом билирубина;
- ни одна пара признаков не превысила порог 0.75–0.80, мультиколлинеарности нет.

## 5. Feature Engineering и медицинские шкалы

### 5.1. Бинарный флаг печёночной недостаточности

In [ ]:
df['is_liver_failure'] = (
    (df['Bilirubin'] > 6.45) & 
    ((df['Ascites'] == 'Y') | (df['Edema'].isin(['Y', 'S']))) & 
    (df['Albumin'] < 2.72)
).astype(int)

print('Пациентов с критическим состоянием:', df['is_liver_failure'].sum())
print('Из них с летальным исходом:', df.loc[df['is_liver_failure'] == 1, 'Status'].eq('D').sum())

Найдено 77 таких пациентов — все с летальным исходом. Признак добавлен как детерминированный маркер высокого риска.

### 5.2. Mayo PBC Score

In [ ]:
age_years = df['Age'] / 365.25
edema_mayo = df['Edema'].map({'N': 0.0, 'S': 0.5, 'Y': 1.0}).astype(float)
mayo_score = (
    0.039 * age_years +
    0.871 * np.log(df['Bilirubin']) +
    0.859 * edema_mayo -
    2.53 * np.log(df['Albumin']) +
    2.38 * np.log(df['Prothrombin'])
)
high_risk = mayo_score >= 7.8
print('Пациентов высокого риска (Mayo >= 7.8):', high_risk.sum())
print('Из них умерло:', df.loc[high_risk, 'Status'].eq('D').sum())

Порог ≥ 7.8 указывает на высокий риск. Выделено 228 пациентов высокого риска, из них 218 (95.6%) умерли. Шкала хорошо ловит летальные исходы, но не захватывает пациентов с трансплантацией.

### 5.3. ALBI Score

In [ ]:
bili_umol_l = df['Bilirubin'] * 17.1
albumin_g_l = df['Albumin'] * 10
albi_score = np.log10(bili_umol_l) * 0.66 - 0.085 * albumin_g_l

albi_grade = pd.cut(
    albi_score,
    bins=[-np.inf, -2.60, -1.39, np.inf],
    labels=[1, 2, 3]
).astype(int)

print(pd.crosstab(albi_grade, df['Status']))

Стратификация:

- степень 1 (≤ -2.60) — удовлетворительная функция печени, 88.3% выжили;
- степень 2 (-2.60 до -1.39) — умеренная декомпенсация, сюда попала большая часть класса `CL`;
- степень 3 (> -1.39) — выраженная декомпенсация, 84.8% умерли.

Признак хорошо разделяет пациентов по тяжести.

## 6. Гипотезы для этапа моделирования

**Трансформация распределений.** Логарифмирование скошенных биохимических показателей (`Prothrombin`, `SGOT`, `Copper`, `Tryglicerides`, `Alk_Phos`, `Bilirubin`, `Cholesterol`) нормализует распределения, улучшит сходимость линейных моделей и снизит влияние выбросов на функцию потерь.

**Интегральные клинические шкалы.** Включение `Mayo PBC Score`, `ALBI score` и `is_liver_failure` должно повысить качество предсказания по сравнению с изолированными лабораторными значениями, потому что эти признаки агрегируют взаимосвязанные биомаркеры в клинически интерпретируемые метрики.

**Кодирование категориальных признаков.** Для `Edema` в линейных моделях лучше использовать One-Hot Encoding, а не порядковое кодирование — иначе модель будет воспринимать N < S < Y как числовую прогрессию, что не соответствует реальной тяжести. Для деревьев можно оставить как есть. `Stage` — аналогично.

**Проблема миноритарного класса.** Без специальных приёмов (взвешивание классов, калибровка порогов, ресемплинг) модель будет систематически недопредсказывать класс `CL` — recall по нему будет близок к нулю, несмотря на высокий общий accuracy за счёт класса `C`.

**Неинформативность `id` и `Drug`.** Идентификатор пациента не должен участвовать в обучении. Признак `Drug` имеет практически нулевую корреляцию с целевой переменной (0.01) и, вероятно, окажется незначимым для модели.

## 7. План предобработки

**Очистка:** удаление частичных дубликатов. Поле `id` сохраняется в отдельную переменную для формирования сабмишна и исключается из признаков, подаваемых в модель.

**Генерация признаков:**
- конвертация возраста в годы: `Age_years = Age / 365.25`;
- бинарный флаг `is_liver_failure`;
- расчёт `Mayo PBC Score` и `ALBI score`.

**Математическая трансформация:** `log1p` к `Prothrombin`, `SGOT`, `Copper`, `Tryglicerides`, `Alk_Phos`, `Bilirubin`, `Cholesterol`.

**Кодирование категориальных признаков:**
- бинарное (0/1): `Sex`, `Drug`, `Ascites`, `Spiders`, `Hepatomegaly`;
- One-Hot Encoding: `Edema` (столбцы `Edema_N`, `Edema_S`, `Edema_Y`);
- `Stage` — оставить как есть для деревьев, OHE для линейных моделей.

**Оптимизация типов:**
- целочисленные признаки через `pd.to_numeric(downcast='integer')`;
- вещественные через `pd.to_numeric(downcast='float')`;
- категориальные с низкой уникальностью (< 20%) — в тип `category`.

**Валидация и метрики:**
- StratifiedKFold со стратификацией по `Status`, чтобы в каждом фолде были примеры `CL`;
- основные метрики: LogLoss (для качества вероятностей в сабмишне) и F1-macro (для общей оценки по трём классам).